In [3]:
import os
import numpy as np
import pandas as pd

# ============ 0. PARÁMETROS (lo que puedes mover) ============
data_folder = 'BACI_HS92_V202601'
YEARS = [2021, 2022, 2023]        # años que se promedian
MIN_EXPORT = 100                  # miles de USD por año (100 = USD 100 mil)
MIN_COUNTRY_TOTAL = 1_000_000     # miles de USD (= USD 1.000 millones)
BASKET_PREFIXES = ['8401', '2844', '8109']  # VERIFÍCALOS con lo que imprime abajo

# ============ 1. Diccionarios ============
print('Cargando diccionarios de países y productos...')
df_countries = pd.read_csv(os.path.join(data_folder, 'country_codes_V202601.csv'))
df_products = pd.read_csv(os.path.join(data_folder, 'product_codes_HS92_V202601.csv'))

df_products['code'] = pd.to_numeric(df_products['code'], errors='coerce')
df_products = df_products.dropna(subset=['code'])
df_products['code'] = df_products['code'].astype(int)

country_map = dict(zip(df_countries['country_code'], df_countries['country_iso3']))
product_map = dict(zip(df_products['code'], df_products['description']))

# ============ 2. Cargar varios años y promediar ============
frames = []
for y in YEARS:
    f = os.path.join(data_folder, f'BACI_HS92_Y{y}_V202601.csv')
    print(f'Procesando {f}...')
    d = pd.read_csv(f, usecols=['i', 'k', 'v'])   # i=exportador, k=producto, v=miles USD
    frames.append(d.groupby(['i', 'k'])['v'].sum())

# suma de los años / número de años = exportación anual promedio
df_trade = (pd.concat(frames).groupby(level=[0, 1]).sum() / len(YEARS)).reset_index()
df_trade['country'] = df_trade['i'].map(country_map)
df_trade = df_trade.dropna(subset=['country'])

# ============ 3. Matriz País x Producto ============
print('Construyendo matriz de exportaciones...')
X = df_trade.pivot_table(index='country', columns='k', values='v',
                         aggfunc='sum', fill_value=0.0)

X = X.loc[X.sum(axis=1) > MIN_COUNTRY_TOTAL]   # países con exportaciones relevantes
X = X.loc[:, X.sum(axis=0) > 0]                # quita productos que nadie exporta (adiós warning)
print(f'Matriz limpia: {X.shape[0]} países x {X.shape[1]} productos')

# ============ 4. RCA + umbral mínimo ============
X_vals = X.values
country_totals = X_vals.sum(axis=1, keepdims=True)
product_totals = X_vals.sum(axis=0, keepdims=True)
world_total = X_vals.sum()

RCA = (X_vals / country_totals) / (product_totals / world_total)

# Competitivo = RCA >= 1 Y además exporta un mínimo real (evita "un solo envío")
M = ((RCA >= 1.0) & (X_vals >= MIN_EXPORT)).astype(int)

# ============ 5. Proximidad entre productos (Phi) ============
print('Calculando la red de proximidades...')
C = M.T @ M                                   # cuántos países exportan AMBOS productos
ubiq = M.sum(axis=0)                          # cuántos países exportan cada producto
max_ubiq = np.maximum(ubiq[:, None], ubiq[None, :])
Phi = np.divide(C, max_ubiq, out=np.zeros(C.shape, dtype=float), where=max_ubiq != 0)
np.fill_diagonal(Phi, 0.0)

# ============ 6. Canasta nuclear ============
cols = list(X.columns)
basket = [c for c in cols if str(int(c)).zfill(6).startswith(tuple(BASKET_PREFIXES))]
idxs = [cols.index(c) for c in basket]

print('\nCanasta nuclear (verifica que sean los productos que quieres):')
for c, j in zip(basket, idxs):
    print(f'  {c} | exportadores competitivos: {ubiq[j]:3d} | {str(product_map.get(c, "?"))[:70]}')

# Densidad de TODOS los países hacia cada producto de la canasta
den = Phi[:, idxs].sum(axis=0).astype(float)
den[den == 0] = np.nan                        # evita dividir entre 0
Omega = pd.DataFrame((M @ Phi[:, idxs]) / den, index=X.index, columns=basket)
omega_basket = Omega.mean(axis=1)             # densidad promedio hacia la canasta

# Chequeo de cordura: ¿salen los países que uno espera?
print('\nTop 10 países por densidad hacia la canasta nuclear:')
print(omega_basket.sort_values(ascending=False).head(10).round(3))

# ============ 7. Colombia ============
if 'COL' in X.index:
    i_col = list(X.index).index('COL')

    val = omega_basket['COL']
    rank = int(omega_basket.rank(ascending=False)['COL'])
    print(f'\n>>> Densidad de COL hacia la canasta nuclear: {val:.4f}')
    print(f'>>> Puesto {rank} de {len(omega_basket)} países')

    # Línea base: densidad de Colombia hacia CUALQUIER producto
    den_all = Phi.sum(axis=0).astype(float)
    den_all[den_all == 0] = np.nan
    omega_col_all = (M[i_col] @ Phi) / den_all
    pct = np.nanmean(omega_col_all < val) * 100
    print(f'>>> Promedio de COL hacia cualquier producto: {np.nanmean(omega_col_all):.4f}')
    print(f'>>> La canasta nuclear está en el percentil {pct:.0f} de los productos de Colombia')

    # Detalle por código
    print('\nDensidad de COL por código de la canasta:')
    print(Omega.loc['COL'].round(4).to_string())

    # Vecinos más cercanos a la canasta (sin contar la canasta misma)
    phi_basket = Phi[:, idxs].mean(axis=1)
    phi_basket[idxs] = -1
    top = np.argsort(phi_basket)[::-1][:15]

    df_results = pd.DataFrame({
        'Código HS': [cols[j] for j in top],
        'Proximidad': [round(phi_basket[j], 3) for j in top],
        'Colombia Competitiva': ['SÍ' if M[i_col, j] == 1 else 'NO' for j in top],
        'Descripción': [str(product_map.get(cols[j], ''))[:60] for j in top],
    })
    print('\nTop 15 productos más cercanos a la canasta nuclear:')
    try:
        display(df_results)
    except NameError:
        print(df_results.to_string(index=False))
else:
    print("No se encontró 'COL' en la lista de países.")
    print(omega_basket.rank(ascending=False)[['RUS', 'KOR', 'CAN', 'IND', 'ARG', 'BRA']])

Cargando diccionarios de países y productos...
Procesando BACI_HS92_V202601/BACI_HS92_Y2021_V202601.csv...
Procesando BACI_HS92_V202601/BACI_HS92_Y2022_V202601.csv...
Procesando BACI_HS92_V202601/BACI_HS92_Y2023_V202601.csv...
Construyendo matriz de exportaciones...
Matriz limpia: 164 países x 4656 productos
Calculando la red de proximidades...

Canasta nuclear (verifica que sean los productos que quieres):
  284410 | exportadores competitivos:   9 | Uranium: natural uranium and its compounds, alloys, dispersions (inclu
  284420 | exportadores competitivos:   8 | Uranium: enriched in U235, plutonium, their compounds, alloys dispersi
  284430 | exportadores competitivos:  10 | Uranium: depleted in U235, thorium, their compounds, alloys, dispersio
  284440 | exportadores competitivos:  16 | Radioactive elements, isotopes, compounds, n.e.s. in heading no. 2844 
  284450 | exportadores competitivos:   6 | Spent (irradiated) fuel elements (cartridges) of nuclear reactors
  810910 | exportad

,Código HS,Proximidad,Colombia Competitiva,Descripción
0,854049,0.348,NO,"Tubes: microwave tubes, other than magnetrons ..."
1,810890,0.339,NO,"Titanium: other than unwrought, n.e.s. in head..."
2,842710,0.323,NO,Fork-lift and other works trucks: fitted with ...
3,470610,0.303,NO,Pulp: cotton linters pulp
4,740610,0.301,NO,Copper: powders of non-lamellar structure
5,280519,0.298,NO,Alkali metals: other than sodium
6,750712,0.295,NO,"Nickel: tubes and pipes, of nickel alloys"
7,290323,0.295,NO,Unsaturated chlorinated derivatives of acyclic...
8,291712,0.295,NO,Acids: acyclic polycarboxylic acids: adipic ac...
9,281310,0.294,NO,Carbon disulphide
